# Clusters

## All Imports

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np
import scipy.cluster.hierarchy as shc
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import normalize, PowerTransformer
from sklearn.pipeline import Pipeline
from yellowbrick.cluster import KElbowVisualizer


In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']


In [ ]:
df = pd.read_csv("../data/pokedex_post_eda.csv")
df.info()


In [ ]:
df2 = pd.read_csv("../data/pokedex_description.csv")
df2.head()


## Cluster (HP, Attack, Defense, SP_Attack, SP_defense, Speed):

### Metrics

In [ ]:
stats_column = ['HP', 'Attack', 'Defense', 'SP_Attack', 'SP_Defense', 'Speed', 'Weight', 'Height']
X = df[stats_column]

scaler = PowerTransformer(method='yeo-johnson', standardize=True)
X_scaled = scaler.fit_transform(X)

plt.figure(figsize=(12, 8))
plt.title("Pokémon Dendrogram (Based on Battle Status)")
plt.xlabel("Pokémons (index)")
plt.ylabel("Euclidean Distance (Dissimilarity)")

dend = shc.dendrogram(shc.linkage(X_scaled, method='ward'),
                      truncate_mode='lastp',
                      p=30,
                      leaf_rotation=90.,
                      leaf_font_size=10.,
                      show_contracted=True)

plt.axhline(y=23, color='r', linestyle='--') 
plt.show()

In [ ]:
K_range = range(2, 15)

inertia = []
silhouette_avg = []
davies_bouldin = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

    silhouette_avg.append(silhouette_score(X_scaled, kmeans.labels_))

    davies_bouldin.append(davies_bouldin_score(X_scaled, kmeans.labels_))

fig, ax = plt.subplots(1, 3, figsize=(24, 6))

ax[0].plot(K_range, inertia, 'bo--')
ax[0].set_title('Elbow Method (Inertia - Lower is better)')
ax[0].set_xlabel('Number of Clusters (k)')
ax[0].set_ylabel('Inertia (Sum of Squared Errors)')
ax[0].grid(True)

ax[1].plot(K_range, silhouette_avg, 'ro--')
ax[1].set_title('Silhouette Score (Higher is better)')
ax[1].set_xlabel('Number of Clusters (k)')
ax[1].set_ylabel('Silhouette Score')
ax[1].grid(True)

ax[2].plot(K_range, davies_bouldin, 'go--')
ax[2].set_title('Davies-Bouldin Index (Lower is better)')
ax[2].set_xlabel('Number of Clusters (k)')
ax[2].set_ylabel('Davies-Bouldin Score')
ax[2].grid(True)

plt.tight_layout()
plt.show()

#### Params

In [ ]:
n_clusters = 7

### Group

In [ ]:
stats_column = ['HP', 'Attack', 'Defense', 'SP_Attack', 'SP_Defense', 'Speed', 'Weight', 'Height']
X = df[stats_column]

pipeline = Pipeline([
    ('scaler', PowerTransformer(method='yeo-johnson', standardize=True)),
    ('clustering', KMeans(n_clusters=n_clusters, init='k-means++', n_init=10, random_state=42))
])

df['class'] = pipeline.fit_predict(X)

avg_profile = df.groupby('class')[stats_column].mean()

count = df['class'].value_counts()

print("Average Status per Cluster:")
print(avg_profile)
print("\nNumber of Pokémons for Cluster:")
print(count)

for i in range(n_clusters):
    print(f"\nCluster {i} Samples:")
    print(df[df['class'] == i]['Name'].head(5).values)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(pipeline.named_steps['scaler'].transform(X))

df_plot = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_plot['Cluster'] = df['class']

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_plot, x='PC1', y='PC2', hue='Cluster', palette='viridis', s=60)
plt.title("Visualisation of Pokémon Clusters (Reduced via PCA)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.show()

In [ ]:
cross_cat = pd.crosstab(df['class'], df['Category'], normalize='index') * 100

for i in range(n_clusters):
    top_cat = cross_cat.loc[i].idxmax()
    prop_cat = cross_cat.loc[i].max()
    avg_get = df[df['class'] == i]['Get_Rate'].mean()
    
    n_legendaries = df[(df['class'] == i) & (df['Category'] == 'Legendary')].shape[0]
    n_megas = df[(df['class'] == i) & (df['Mega_Evolution_Flag'].notna())].shape[0]
    n_mythicals = df[(df['class'] == i) & (df['Category'] == 'Mythical')].shape[0]
    n_semi = df[(df['class'] == i) & (df['Category'] == 'Semi-Legendary')].shape[0]
    
    print(f"CLUSTER {i}:")
    print(f" - Average Capture Rate: {avg_get:.1f}")
    print(f" - Number of Legendaries: {n_legendaries}")
    print(f" - Number of Mythical: {n_mythicals}")
    print(f" - Number of Semi-Legendaries: {n_semi}")
    print(f" - Number of Megas: {n_megas}")
    print("-" * 30)

In [ ]:
for i in range(n_clusters):
    print(f"--- Cluster {i} Samples ---")
    print(df[df['class'] == i]['Name'].sample(5).values)
    print("-"*30)

## Cluster (HP, Attack, Defense, SP_Attack, SP_defense, Speed, Type):

### Metrics

In [ ]:
stats_column = [
    'HP', 'Attack', 'Defense', 'SP_Attack', 'SP_Defense', 'Speed', 'Weight', 'Height', 
    'Bug', 'Dark', 'Dragon', 'Electric', 'Fairy', 'Fighting', 'Fire', 'Flying', 'Ghost', 
    'Grass', 'Ground', 'Ice', 'Normal', 'Poison', 'Psychic', 'Rock', 'Steel', 'Water'
]
X = df[stats_column]

scaler = PowerTransformer(method='yeo-johnson', standardize=True)
X_scaled = scaler.fit_transform(X)

plt.figure(figsize=(12, 8))
plt.title("Pokémon Dendrogram (Based on Battle Status and Type)")
plt.xlabel("Pokémons (index)")
    'HP', 'Attack', 'Defense', 'SP_Attack', 'SP_Defense', 'Speed', 'Weight', 'Height', 
plt.ylabel("Euclidean Distance (Dissimilarity)")

dend = shc.dendrogram(shc.linkage(X_scaled, method='ward'),
                      truncate_mode='lastp',
                      p=30,
                      leaf_rotation=90.,
                      leaf_font_size=10.,
                      show_contracted=True)

plt.axhline(y=18, color='r', linestyle='--') 
plt.show()

In [ ]:
K_range = range(2, 15)

inertia = []
silhouette_avg = []
davies_bouldin = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

    silhouette_avg.append(silhouette_score(X_scaled, kmeans.labels_))

    davies_bouldin.append(davies_bouldin_score(X_scaled, kmeans.labels_))

fig, ax = plt.subplots(1, 3, figsize=(24, 6))

ax[0].plot(K_range, inertia, 'bo--')
ax[0].set_title('Elbow Method (Inertia - Lower is better)')
ax[0].set_xlabel('Number of Clusters (k)')
ax[0].set_ylabel('Inertia (Sum of Squared Errors)')
ax[0].grid(True)

ax[1].plot(K_range, silhouette_avg, 'ro--')
ax[1].set_title('Silhouette Score (Higher is better)')
ax[1].set_xlabel('Number of Clusters (k)')
ax[1].set_ylabel('Silhouette Score')
ax[1].grid(True)

ax[2].plot(K_range, davies_bouldin, 'go--')
ax[2].set_title('Davies-Bouldin Index (Lower is better)')
ax[2].set_xlabel('Number of Clusters (k)')
ax[2].set_ylabel('Davies-Bouldin Score')
ax[2].grid(True)

plt.tight_layout()
plt.show()

#### Params

In [ ]:
n_clusters = 7


### Group

In [ ]:
numeric_features = ['HP', 'Attack', 'Defense', 'SP_Attack', 'SP_Defense', 'Speed', 'Weight', 'Height']

binary_features = [
    'Bug', 'Dark', 'Dragon', 'Electric', 'Fairy', 'Fighting', 'Fire', 'Flying', 'Ghost', 
    'Grass', 'Ground', 'Ice', 'Normal', 'Poison', 'Psychic', 'Rock', 'Steel', 'Water'
]

features_to_use = numeric_features + binary_features
X = df[features_to_use]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', PowerTransformer(method='yeo-johnson', standardize=True), numeric_features),
        ('cat', 'passthrough', binary_features)
    ])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clustering', KMeans(n_clusters=n_clusters, init='k-means++', n_init=10, random_state=42))
])

df['class'] = pipeline.fit_predict(X)

avg_profile = df.groupby('class')[stats_column].mean()

count = df['class'].value_counts()

print("Average Status per Cluster:")
print(avg_profile)
print("\nNumber of Pokémons for Cluster:")
print(count)

for i in range(n_clusters):
    print(f"\nCluster {i} Samples:")
    print(df[df['class'] == i]['Name'].head(5).values)

In [ ]:
pca = PCA(n_components=2)

X_transformed = pipeline.named_steps['preprocessor'].transform(X)

X_pca = pca.fit_transform(X_transformed)

df_plot = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_plot['Cluster'] = df['class']

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_plot, x='PC1', y='PC2', hue='Cluster', palette='viridis', s=60)
plt.title("Visualisation of Pokémon Clusters (K-Means + PCA)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.show()

In [ ]:
cross_cat = pd.crosstab(df['class'], df['Category'], normalize='index') * 100

for i in range(n_clusters):
    top_cat = cross_cat.loc[i].idxmax()
    prop_cat = cross_cat.loc[i].max()
    avg_get = df[df['class'] == i]['Get_Rate'].mean()
    
    n_legendaries = df[(df['class'] == i) & (df['Category'] == 'Legendary')].shape[0]
    n_megas = df[(df['class'] == i) & (df['Mega_Evolution_Flag'].notna())].shape[0]
    n_mythicals = df[(df['class'] == i) & (df['Category'] == 'Mythical')].shape[0]
    n_semi = df[(df['class'] == i) & (df['Category'] == 'Semi-Legendary')].shape[0]
    
    print(f"CLUSTER {i}:")
    print(f" - Average Capture Rate: {avg_get:.1f}")
    print(f" - Number of Legendaries: {n_legendaries}")
    print(f" - Number of Mythical: {n_mythicals}")
    print(f" - Number of Semi-Legendaries: {n_semi}")
    print(f" - Number of Megas: {n_megas}")
    print("-" * 30)

In [ ]:
for i in range(n_clusters):
    print(f"--- Cluster {i} Samples ---")
    print(df[df['class'] == i]['Name'].sample(5).values)
    print("-"*30)

## Cluster Description

### Embeddings

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2') 


In [ ]:
embeddings = model.encode(df2["info"].tolist())
df2['embeddings'] = list(embeddings)


In [ ]:
df2.head()


In [ ]:
df2["embeddings"][0].size


In [ ]:
X = np.vstack(df2['embeddings'].values)
X = normalize(X)


### K-Definition

In [ ]:
visualizer = KElbowVisualizer(KMeans(random_state=42), k=(2,15))
visualizer.fit(X) 
visualizer.show()


### Clusters

In [ ]:
kmeans = KMeans(n_clusters=8, random_state=42)
kmeans.fit(X)


In [ ]:
df2["cluster"]=kmeans.labels_.astype(str)


In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(X)

df2['pca_x'] = pca_result[:, 0]
df2['pca_y'] = pca_result[:, 1]


In [ ]:
centroides_original = kmeans.cluster_centers_
centroides_2d = pca.transform(centroides_original)

df_centroides = pd.DataFrame(centroides_2d, columns=['pca_x', 'pca_y'])
df_centroides['cluster'] = df_centroides.index.astype(str)

fig = px.scatter(
    df2, 
    x='pca_x', 
    y='pca_y', 
    color='cluster',
    hover_data=['name'],
    title='Clusters Pokémon + Centroides',
    opacity=0.6
)

fig.add_trace(
    go.Scatter(
        x=df_centroides['pca_x'],
        y=df_centroides['pca_y'],
        mode='markers',
        name='Centroides',
        marker=dict(
            color='black',
            size=10,
            symbol='x',
            line=dict(width=2)
        ),
        text=[f"Cluster Center {i}" for i in range(len(df_centroides))],
        hoverinfo='text'
    )
)

fig.show()


In [ ]:
for id in sorted(df2['cluster'].unique()):
    print(f"\nCLUSTER {id}:")
    group = df2[df2['cluster'] == id]
    
    sample = group.sample(n=min(3, len(group)), random_state=42)
    
    for name, description in zip(sample['name'], sample['info']):
        print(f"   - {name}: {description}")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df2['cluster'] = df2['cluster'].astype(str)
grouped_text = df2.groupby('cluster')['info'].apply(lambda x: ' '.join(x)).reset_index()

my_stop_words = list(ENGLISH_STOP_WORDS) + ['pokémon', 'trainer', '000']

tfidf = TfidfVectorizer(
    stop_words=my_stop_words, # Aqui acontece a mágica
    max_features=500
)
tfidf_matrix = tfidf.fit_transform(grouped_text['info'])
feature_names = tfidf.get_feature_names_out()

print("Top keywords per text cluster:")
dense = tfidf_matrix.todense()
for i in range(len(grouped_text)):
    cluster_id = grouped_text.iloc[i]['cluster']
    top_indices = dense[i].argsort().tolist()[0][-10:][::-1]
    top_words = [feature_names[ind] for ind in top_indices]
    print(f"Cluster {cluster_id}: {', '.join(top_words)}")